# Agentic AI Systems — Worklet 15
# Executive Decision Support System
## Lab 4 · Memory & Retrieval — state vs stored

**Phase A · Foundations** &nbsp;|&nbsp; **Goal:** separate short-term run state from long-term stored knowledge.

---

### What Lab 4 contributes to Worklet 15

| | |
|---|---|
| **Inputs** | Institutional KPIs & dashboards; history of prior decision briefs; the current draft. |
| **Outputs** | Tag- and similarity-based retrieval. |
| **Agents / nodes** | Retrieval agent; the foundation of later screening. |
| **Deliverable** | Retrieval over the store + history with tag filters and embedding similarity. |
| **Acceptance** | **For a given task, retrieval returns correctly-tagged candidates and similar prior items.** |
| **Pitfall to avoid** | Don't conflate run state with the archive — mixing them corrupts the baseline. |

> Chapter 15 pseudocode:
> ```
> state.draft = candidate                            # short-term
> hits  = retrieve(store, tags)                      # long-term
> prior = retrieve(history, embed(candidate), k=5)   # similarity
> ```

### Why this lab matters to *this* worklet

Every run so far has started from nothing. It could tell leadership that placement is **below target** — but not whether it is **getting better or worse**, and not whether the option it is about to propose was already tried and already failed.

Both need a baseline that outlives the run. That is what the archive is.

### Two memories, two lifetimes

| | `RunState` | `Archive` |
|---|---|---|
| Lifetime | one run | across years |
| Mutability | freely mutable | **append-only** |
| Reads | direct | **copies**, so a reader cannot corrupt it |
| Authority | never a source of truth | the institutional baseline |
| Entry | anything | only an **approved** brief, audited |

### How to run

Runs top to bottom with **no API key and no network**. The similarity index is a TF-IDF + cosine implementation written out in full in Cell 5 — every score is reproducible by hand. Cell 9 executes the rubric as assertions.

---

### Cell 1 · The shared foundation

The Labs 1–2 engine, unchanged, written to `edss_core.py` so this notebook runs standalone.

*(Collapse this cell — prior work, restated only for self-containment.)*

In [1]:
%%writefile edss_core.py
"""edss_core — shared foundation for the Worklet 15 lab notebooks.

Worklet 15 · Executive Decision Support System · Agentic AI (26ECAC401)

This is the Lab 1 + Lab 2 foundation, unchanged: the institutional data, the
typed tool contracts, the single ToolRouter choke point, the Connector, and the
validation gate. Labs 3-8 build on top of it rather than restating it, so every
notebook works against the same tools and the same numbers.

Written to disk by the first code cell of each lab notebook, so each notebook
runs standalone in Colab with no setup.
"""

from __future__ import annotations

import csv
import io
import json
import math
import re
import time
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

# 1 · Institutional data (identical to Lab 2, Cell 3)
# =============================================================================

REPORTING_PERIOD = "academic year 2025-26"

DEPARTMENT_REPORTS_CSV = """dept_code,dept_name,category,students_appeared,students_passed,co_po_attainment,sanctioned_faculty,publications,citations,grants_lakh,eligible_students,placed_students,total_package_lpa,interns_completed,budget_allocated_lakh,budget_spent_lakh,figure_status
CSE,Computer Science & Engineering,circuit,240,231,2.71,34,58,402,96.5,228,212,2247.2,205,420,393,final
ECE,Electronics & Communication,circuit,180,168,2.58,26,34,221,41.0,172,143,1058.2,149,310,276,final
ISE,Information Science & Engineering,circuit,120,116,2.66,18,21,140,22.5,114,103,947.6,96,240,221,final
EEE,Electrical & Electronics,circuit,120,109,2.41,17,16,88,18.0,116,86,524.6,84,210,181,final
MECH,Mechanical Engineering,core,180,158,2.33,24,19,95,27.5,174,111,599.4,102,300,247,final
CIVIL,Civil Engineering,core,120,101,2.19,16,11,47,12.0,118,62,285.2,61,190,149,final
CHEM,Chemical Engineering,core,60,54,2.28,11,14,76,15.5,58,33,168.3,34,140,118,final
BT,Biotechnology,core,60,55,2.35,10,12,63,9.5,57,29,142.1,30,130,102,final
"""

NUMERIC_COLUMNS = {
    "students_appeared": int, "students_passed": int, "co_po_attainment": float,
    "sanctioned_faculty": int, "publications": int, "citations": int,
    "grants_lakh": float, "eligible_students": int, "placed_students": int,
    "total_package_lpa": float, "interns_completed": int,
    "budget_allocated_lakh": float, "budget_spent_lakh": float,
}

KPI_DEFINITIONS: Dict[str, Dict[str, Any]] = {
    "KPI-A1": {"name": "Institution pass percentage", "domain": "Academic",
               "method": "ratio", "numerator": "students_passed",
               "denominator": "students_appeared", "scale": 100, "unit": "%",
               "formula": "sum(students_passed) / sum(students_appeared) x 100",
               "target": 92.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Academic ERP"},
    "KPI-A2": {"name": "CO/PO attainment index", "domain": "Academic",
               "method": "weighted_mean", "value_field": "co_po_attainment",
               "weight_field": "students_appeared", "unit": "index 0-3",
               "formula": "sum(co_po_attainment x students_appeared) / sum(students_appeared)",
               "target": 2.50, "target_source": "NAAC criterion 2",
               "direction": "higher_is_better", "source_system": "Academic ERP (OBE module)"},
    "KPI-R1": {"name": "Publications per faculty", "domain": "Research",
               "method": "ratio", "numerator": "publications",
               "denominator": "sanctioned_faculty", "scale": 1, "unit": "per faculty",
               "formula": "sum(publications) / sum(sanctioned_faculty)",
               "target": 1.50, "target_source": "NIRF Research & Professional Practice",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-R2": {"name": "Citations per faculty", "domain": "Research",
               "method": "ratio", "numerator": "citations",
               "denominator": "sanctioned_faculty", "scale": 1, "unit": "per faculty",
               "formula": "sum(citations) / sum(sanctioned_faculty)",
               "target": 9.00, "target_source": "NIRF Research & Professional Practice",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-R3": {"name": "Grants sanctioned", "domain": "Research",
               "method": "sum", "value_field": "grants_lakh", "unit": "INR lakh",
               "formula": "sum(grants_lakh)",
               "target": 300.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Research repository"},
    "KPI-P1": {"name": "Placement percentage", "domain": "Placement",
               "method": "ratio", "numerator": "placed_students",
               "denominator": "eligible_students", "scale": 100, "unit": "%",
               "formula": "sum(placed_students) / sum(eligible_students) x 100",
               "target": 80.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-P2": {"name": "Average package", "domain": "Placement",
               "method": "ratio", "numerator": "total_package_lpa",
               "denominator": "placed_students", "scale": 1, "unit": "LPA",
               "formula": "sum(total_package_lpa) / sum(placed_students)",
               "target": 8.00, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-P3": {"name": "Internship completion", "domain": "Placement",
               "method": "ratio", "numerator": "interns_completed",
               "denominator": "eligible_students", "scale": 100, "unit": "%",
               "formula": "sum(interns_completed) / sum(eligible_students) x 100",
               "target": 85.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Placement Cell tracker"},
    "KPI-F1": {"name": "Budget utilisation", "domain": "Finance",
               "method": "ratio", "numerator": "budget_spent_lakh",
               "denominator": "budget_allocated_lakh", "scale": 100, "unit": "%",
               "formula": "sum(budget_spent_lakh) / sum(budget_allocated_lakh) x 100",
               "target": 90.0, "target_source": "Strategic Plan 2024-27",
               "direction": "higher_is_better", "source_system": "Finance & Accounts ledger",
               "figure_status": "provisional",
               "caveat": "Q3-Q4 of 2025-26 are unreconciled; the Finance Office closes "
                         "books in August."},
}

BENCHMARK_API_DB = {
    "KPI-R1": {"peer_median": 1.82, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
    "KPI-R2": {"peer_median": 11.40, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
    "KPI-P1": {"peer_median": 78.60, "cohort": "NIRF Engineering rank band 51-100", "year": "2025"},
}

DOMAINS = ["Academic", "Research", "Placement", "Finance"]
DOMAIN_KPIS = {d: [k for k, v in KPI_DEFINITIONS.items() if v["domain"] == d] for d in DOMAINS}

# Registrar-authorised read access, per Gate 1 of the hybrid execution methodology.
ACCESS_REGISTER = {
    "Academic":  {"source_system": "Academic ERP — examination & OBE records",
                  "read_access": "granted by Registrar on 2026-06-01", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["pass percentage", "CO/PO attainment", "curriculum compliance"]},
    "Research":  {"source_system": "Research & Publications repository (Scopus-synced)",
                  "read_access": "granted by Registrar on 2026-06-01", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["publications", "citations", "grants sanctioned"]},
    "Placement": {"source_system": "Placement Cell tracker",
                  "read_access": "granted by Registrar on 2026-06-04", "figure_status": "final",
                  "periods": ["2023-24", "2024-25", "2025-26"],
                  "metrics": ["placement percentage", "average package",
                              "internship completion"],
                  "caveat": "3 provisional offers are excluded until the Placement Cell "
                            "verifies them."},
    "Finance":   {"source_system": "Finance & Accounts ledger",
                  "read_access": "granted by Registrar on 2026-06-07",
                  "figure_status": "provisional",
                  # 2025-26 is deliberately absent: the books are not closed, and the
                  # Lab 1 planner must notice that rather than plan over it.
                  "periods": ["2023-24", "2024-25"],
                  "metrics": ["budget utilisation", "revenue", "grant funds"],
                  "caveat": "Q3-Q4 of 2025-26 are unreconciled; the Finance Office closes "
                            "books in August."},
}


# =============================================================================
# 2 · Tool contracts (identical to Lab 2, Cell 4)
# =============================================================================

TOOL_SCHEMAS: Dict[str, Dict[str, Any]] = {
    "read_department_reports": {
        "description": "Read per-department metric rows for one reporting period.",
        "parameters": {"type": "object", "properties": {
            "period": {"type": "string"},
            "departments": {"type": "array", "items": {"type": "string"}},
            "category": {"type": "string", "enum": ["circuit", "core"]}},
            "required": ["period"]},
        "returns": "{rows, row_count, source_system} | {error}"},
    "read_kpi_definitions": {
        "description": "Read approved KPI definitions: formula, unit, target, authority.",
        "parameters": {"type": "object", "properties": {
            "kpi_ids": {"type": "array", "items": {"type": "string"}}}, "required": []},
        "returns": "{definitions, count} | {error}"},
    "read_external_benchmark": {
        "description": "Published peer median for a KPI. Coverage is partial.",
        "parameters": {"type": "object", "properties": {"kpi_id": {"type": "string"}},
                       "required": ["kpi_id"]},
        "returns": "{kpi_id, peer_median, cohort, year} | {error}"},
    "filter_rows": {
        "description": "Filter department rows by an exact field value; errors on empty.",
        "parameters": {"type": "object", "properties": {
            "rows": {"type": "array", "items": {"type": "object"}},
            "field": {"type": "string"}, "value": {"type": "string"}},
            "required": ["rows", "field", "value"]},
        "returns": "{rows, row_count, filter} | {error}"},
    "compute_kpi": {
        "description": "Compute one approved KPI over department rows, with evidence.",
        "parameters": {"type": "object", "properties": {
            "kpi_id": {"type": "string"},
            "rows": {"type": "array", "items": {"type": "object"}}},
            "required": ["kpi_id", "rows"]},
        "returns": "{kpi_id, value, unit, formula, evidence, source_system} | {error}"},
    "compare_to_target": {
        "description": "Compare a KPI value to its approved target; flags rounding risk.",
        "parameters": {"type": "object", "properties": {
            "kpi_id": {"type": "string"}, "value": {"type": "number"}},
            "required": ["kpi_id", "value"]},
        "returns": "{value, target, variance, status, rounding_warning} | {error}"},
    "compute_variance": {
        "description": "Difference between a KPI value and any reference figure.",
        "parameters": {"type": "object", "properties": {
            "value": {"type": "number"}, "reference": {"type": "number"},
            "label": {"type": "string"}}, "required": ["value", "reference", "label"]},
        "returns": "{variance, abs_variance, pct_of_reference} | {error}"},
    "draft_notification": {
        "description": "DRAFT a leadership notification. Never sends. Sources required.",
        "parameters": {"type": "object", "properties": {
            "audience": {"type": "string"}, "subject": {"type": "string"},
            "findings": {"type": "array", "items": {"type": "object"}}},
            "required": ["audience", "subject", "findings"]},
        "returns": "{status: 'DRAFT - NOT SENT', body, requires_approval_from} | {error}"},
}


def _typed_rows(text: str) -> List[Dict[str, Any]]:
    rows = []
    for raw in csv.DictReader(io.StringIO(text)):
        row = dict(raw)
        for col, caster in NUMERIC_COLUMNS.items():
            if col in row:
                try:
                    row[col] = caster(row[col])
                except (TypeError, ValueError):
                    return [{"error": f"column '{col}' is not {caster.__name__}"}]
        rows.append(row)
    return rows


# Optional demo latency, so the parallel/sequential distinction in the run
# timeline is visible. Off by default; never affects any computed figure.
SOURCE_LATENCY_SECONDS = 0.0


def read_department_reports(period: str, departments=None, category=None) -> Dict[str, Any]:
    if period != REPORTING_PERIOD:
        return {"error": f"no department report export for '{period}'; the only loaded "
                         f"period is '{REPORTING_PERIOD}'"}
    if SOURCE_LATENCY_SECONDS:
        time.sleep(SOURCE_LATENCY_SECONDS)
    rows = _typed_rows(DEPARTMENT_REPORTS_CSV)
    if rows and "error" in rows[0]:
        return {"error": rows[0]["error"]}
    if departments:
        known = {r["dept_code"] for r in rows}
        unknown = [d for d in departments if d not in known]
        if unknown:
            return {"error": f"unknown department code(s): {unknown}; known: {sorted(known)}"}
        rows = [r for r in rows if r["dept_code"] in departments]
    if category:
        rows = [r for r in rows if r["category"] == category]
        if not rows:
            return {"error": f"no departments in category '{category}'"}
    if not rows:
        return {"error": "empty department report export"}
    return {"rows": rows, "row_count": len(rows), "period": period,
            "source_system": "Department reports export (Registrar-authorised read)"}


def read_kpi_definitions(kpi_ids=None) -> Dict[str, Any]:
    if kpi_ids:
        unknown = [k for k in kpi_ids if k not in KPI_DEFINITIONS]
        if unknown:
            return {"error": f"unknown KPI id(s): {unknown}"}
        defs = {k: KPI_DEFINITIONS[k] for k in kpi_ids}
    else:
        defs = dict(KPI_DEFINITIONS)
    return {"definitions": defs, "count": len(defs),
            "source_system": "Institutional KPI register (approved definitions)"}


def read_external_benchmark(kpi_id: str) -> Dict[str, Any]:
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    record = BENCHMARK_API_DB.get(kpi_id)
    if record is None:
        return {"error": f"no published external benchmark for {kpi_id} "
                         f"({KPI_DEFINITIONS[kpi_id]['name']}) in the benchmark API"}
    return {"kpi_id": kpi_id, **record, "source_system": "External benchmark API (NIRF cohort)"}


def filter_rows(rows, field: str, value: str) -> Dict[str, Any]:
    if not rows:
        return {"error": "filter_rows received an empty row set"}
    if field not in rows[0]:
        return {"error": f"unknown field '{field}'; available: {sorted(rows[0])}"}
    subset = [r for r in rows if str(r[field]) == str(value)]
    if not subset:
        return {"error": f"filter {field}={value!r} matched 0 of {len(rows)} rows — "
                         f"refusing to return an empty subset"}
    return {"rows": subset, "row_count": len(subset), "filter": f"{field}=={value}",
            "filtered_from": len(rows)}


def compute_kpi(kpi_id: str, rows: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Generic: the arithmetic comes from the KPI definition, not from this code."""
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    if not rows:
        return {"error": f"cannot compute {kpi_id} over an empty row set"}
    d = KPI_DEFINITIONS[kpi_id]
    method, evidence = d["method"], []
    try:
        if method == "ratio":
            num_f, den_f, scale = d["numerator"], d["denominator"], d["scale"]
            num = sum(r[num_f] for r in rows)
            den = sum(r[den_f] for r in rows)
            if den == 0:
                return {"error": f"{kpi_id}: denominator sum({den_f}) is zero"}
            value = num / den * scale
            inputs = {f"sum({num_f})": round(num, 4), f"sum({den_f})": round(den, 4)}
            for r in rows:
                dv = (r[num_f] / r[den_f] * scale) if r[den_f] else None
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[num_f], "denominator": r[den_f],
                                 "numerator_field": num_f, "denominator_field": den_f,
                                 "dept_value": round(dv, 2) if dv is not None else None,
                                 "share_of_denominator_pct": round(r[den_f] / den * 100, 2)})
        elif method == "weighted_mean":
            vf, wf = d["value_field"], d["weight_field"]
            wsum = sum(r[wf] for r in rows)
            if wsum == 0:
                return {"error": f"{kpi_id}: total weight sum({wf}) is zero"}
            value = sum(r[vf] * r[wf] for r in rows) / wsum
            inputs = {f"sum({vf} x {wf})": round(sum(r[vf] * r[wf] for r in rows), 4),
                      f"sum({wf})": wsum}
            for r in rows:
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[vf], "denominator": r[wf],
                                 "numerator_field": vf, "denominator_field": wf,
                                 "dept_value": r[vf],
                                 "share_of_denominator_pct": round(r[wf] / wsum * 100, 2)})
        elif method == "sum":
            vf = d["value_field"]
            value = sum(r[vf] for r in rows)
            inputs = {f"sum({vf})": round(value, 4)}
            for r in rows:
                evidence.append({"dept_code": r["dept_code"], "dept_name": r["dept_name"],
                                 "numerator": r[vf], "denominator": None,
                                 "numerator_field": vf, "denominator_field": None,
                                 "dept_value": r[vf],
                                 "share_of_denominator_pct": round(r[vf] / value * 100, 2)
                                 if value else None})
        else:
            return {"error": f"unsupported KPI method '{method}' for {kpi_id}"}
    except KeyError as missing:
        return {"error": f"{kpi_id}: department rows are missing column {missing}"}

    statuses = {r.get("figure_status", "final") for r in rows}
    figure_status = d.get("figure_status") or ("final" if statuses == {"final"} else "mixed")
    out = {"kpi_id": kpi_id, "name": d["name"], "domain": d["domain"],
           "value": round(value, 4), "unit": d["unit"], "formula": d["formula"],
           "method": method, "n_departments": len(rows),
           "departments": [r["dept_code"] for r in rows],
           "inputs_used": inputs, "evidence": evidence,
           "source_system": d["source_system"], "figure_status": figure_status}
    if d.get("caveat"):
        out["caveat"] = d["caveat"]
    return out


def compare_to_target(kpi_id: str, value: float) -> Dict[str, Any]:
    if kpi_id not in KPI_DEFINITIONS:
        return {"error": f"unknown KPI id '{kpi_id}'"}
    d = KPI_DEFINITIONS[kpi_id]
    target = d["target"]
    variance = value - target
    pct_of_target = value / target * 100 if target else None
    gap_pct = abs(variance) / target * 100 if target else 0
    status = "on target" if variance >= 0 else ("at risk" if gap_pct <= 2.0 else "below target")
    rounding_warning = None
    if variance < 0:
        for places in (0, 1):
            if round(value, places) >= target:
                rounding_warning = (f"{value:.4g} rounds to {round(value, places)} at "
                                    f"{places} decimal place(s), which would read as meeting "
                                    f"the target of {target}. Report the unrounded figure.")
                break
    return {"kpi_id": kpi_id, "name": d["name"], "value": round(value, 4), "target": target,
            "target_source": d["target_source"], "unit": d["unit"],
            "variance": round(variance, 4),
            "pct_of_target": round(pct_of_target, 2) if pct_of_target else None,
            "status": status, "rounding_warning": rounding_warning}


def compute_variance(value: float, reference: float, label: str) -> Dict[str, Any]:
    if reference == 0:
        return {"error": f"cannot compute variance against a zero reference ({label})"}
    variance = value - reference
    return {"value": round(value, 4), "reference": round(reference, 4), "label": label,
            "variance": round(variance, 4), "abs_variance": round(abs(variance), 4),
            "pct_of_reference": round(value / reference * 100, 2)}


def draft_notification(audience: str, subject: str, findings: List[Dict[str, Any]]):
    if not findings:
        return {"error": "refusing to draft a notification with no findings"}
    unsourced = [f for f in findings if not f.get("source")]
    if unsourced:
        return {"error": f"{len(unsourced)} finding(s) carry no source; every figure in a "
                         f"leadership notification must cite where it came from"}
    body = "\n".join(f"  {i}. {f['statement']}\n     source: {f['source']}"
                     for i, f in enumerate(findings, 1))
    return {"status": "DRAFT - NOT SENT", "audience": audience, "subject": subject,
            "body": body, "findings_cited": len(findings),
            "requires_approval_from": "Principal / Registrar (Chapter 15 human gate)",
            "note": "This tool cannot send. Circulation happens only after human approval."}


TOOLS = {
    "read_department_reports": read_department_reports,
    "read_kpi_definitions": read_kpi_definitions,
    "read_external_benchmark": read_external_benchmark,
    "filter_rows": filter_rows,
    "compute_kpi": compute_kpi,
    "compare_to_target": compare_to_target,
    "compute_variance": compute_variance,
    "draft_notification": draft_notification,
}


# =============================================================================
# 3 · Router, connector, validation gate (Lab 2, Cells 5-6)
# =============================================================================

class ToolContractError(Exception):
    """Raised before a call when arguments do not satisfy the tool schema."""


class ToolError(Exception):
    """Raised after a call when the tool returned an explicit error."""


_JSON_TYPES = {"string": str, "number": (int, float), "integer": int,
               "boolean": bool, "array": list, "object": dict}


def _summarise(obj, _depth=0):
    if isinstance(obj, list):
        if len(obj) > 3 and all(isinstance(x, dict) for x in obj):
            return f"[{len(obj)} rows: {', '.join(str(x.get('dept_code', '?')) for x in obj)}]"
        return [_summarise(x, _depth + 1) for x in obj]
    if isinstance(obj, dict):
        return {k: _summarise(v, _depth + 1) for k, v in obj.items()}
    if isinstance(obj, str) and len(obj) > 160:
        return obj[:157] + "..."
    return obj


class ToolRouter:
    """The single choke point. Validates, executes, logs, raises on error."""

    def __init__(self, agent: str = "Coordinator"):
        self.agent = agent
        self.trace: List[Dict[str, Any]] = []
        self.raw_outputs: List[Any] = []

    def validate_input(self, tool: str, kwargs: Dict[str, Any]) -> None:
        if tool not in TOOL_SCHEMAS:
            raise ToolContractError(f"no such tool: '{tool}'")
        schema = TOOL_SCHEMAS[tool]["parameters"]
        props, required = schema["properties"], schema.get("required", [])
        for name in required:
            if name not in kwargs:
                raise ToolContractError(f"{tool}: missing required argument '{name}'")
        for name, val in kwargs.items():
            if name not in props:
                raise ToolContractError(
                    f"{tool}: unknown argument '{name}'; accepts {sorted(props)}")
            spec = props[name]
            expected = _JSON_TYPES.get(spec["type"])
            if expected and not isinstance(val, expected):
                raise ToolContractError(
                    f"{tool}.{name}: expected {spec['type']}, got {type(val).__name__}")
            if spec.get("enum") and val not in spec["enum"]:
                raise ToolContractError(f"{tool}.{name}: {val!r} not in {spec['enum']}")

    def call(self, tool: str, reason: str, **kwargs) -> Dict[str, Any]:
        self.validate_input(tool, kwargs)
        started = time.perf_counter()
        result = TOOLS[tool](**kwargs)
        entry = {"step": len(self.trace) + 1, "agent": self.agent, "tool": tool,
                 "reason": reason, "input": _summarise(kwargs), "output": _summarise(result),
                 "ok": "error" not in result,
                 "ms": round((time.perf_counter() - started) * 1000, 2),
                 "at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        self.trace.append(entry)
        self.raw_outputs.append(result)
        if "error" in result:
            raise ToolError(f"{tool} failed: {result['error']}")
        return result

    def try_call(self, tool: str, reason: str, **kwargs) -> Dict[str, Any]:
        try:
            return self.call(tool, reason, **kwargs)
        except ToolError:
            return dict(TOOLS[tool](**kwargs))


class Connector:
    """One governed surface over every source. No agent touches a source directly."""

    RESOURCES = {
        "department reports":              "read_department_reports",
        "institutional KPIs & dashboards": "read_kpi_definitions",
        "external benchmarks":             "read_external_benchmark",
    }

    def __init__(self, router: ToolRouter):
        self.router = router

    def read(self, resource: str, reason: str, required: bool = True, **params):
        if resource not in self.RESOURCES:
            raise ToolContractError(f"connector exposes no resource '{resource}'; "
                                    f"available: {sorted(self.RESOURCES)}")
        caller = self.router.call if required else self.router.try_call
        return caller(self.RESOURCES[resource], reason, **params)


class Validated:
    def __init__(self, checks: List[tuple], subject: str):
        self.checks, self.subject = checks, subject
        self.failing = [name for name, ok, _ in checks if not ok]
        self.valid = not self.failing

    def __repr__(self):
        return f"<Validated {self.subject}: {'VALID' if self.valid else self.failing}>"


def validate_kpi_result(computed: Dict[str, Any],
                        compared: Optional[Dict[str, Any]] = None) -> Validated:
    ev = computed.get("evidence", [])
    checks = [
        ("value is a finite number",
         isinstance(computed.get("value"), (int, float)) and math.isfinite(computed["value"]),
         f"value={computed.get('value')}"),
        ("the formula used is recorded", bool(computed.get("formula")),
         computed.get("formula", "")),
        ("the figure cites its source system", bool(computed.get("source_system")),
         computed.get("source_system", "")),
        ("raw inputs are recorded", bool(computed.get("inputs_used")),
         str(computed.get("inputs_used", ""))),
        ("every department in the evidence is identified",
         bool(ev) and all(e.get("dept_code") for e in ev), f"{len(ev)} evidence rows"),
        ("evidence covers exactly the departments computed over",
         len(ev) == computed.get("n_departments"), f"{len(ev)} rows"),
        ("provisional figures carry a caveat",
         computed.get("figure_status") == "final" or bool(computed.get("caveat")),
         f"figure_status={computed.get('figure_status')}"),
    ]
    if compared is not None:
        checks.append(("the target names the authority that set it",
                       bool(compared.get("target_source")), compared.get("target_source", "")))
    return Validated(checks, computed.get("kpi_id", "?"))


# =============================================================================


Overwriting edss_core.py


### Cell 2 · The memory split

`RunState` is a scratchpad for one run. `Archive` is the institutional baseline.

The separation is structural, not a naming convention: the archive refuses overwrites, and `get()` returns a deep copy so that a caller who mutates what it read cannot corrupt the store. Cell 8 attacks both.

In [2]:
# =============================================================================
# Cell 2  ·  The memory split — run state vs the archive
# =============================================================================
# Concept 4: short-term state and long-term storage are DIFFERENT THINGS with
# different lifetimes and different owners.
#
#   RunState  — this run only. Mutable, discarded when the run ends. Holds the
#               draft being worked on. Never a source of truth.
#   Archive   — across runs. Append-only, versioned, immutable once written.
#               Holds past decision briefs and closed-year KPI snapshots.
#
# The pitfall this lab grades: "Don't conflate run state with the archive —
# mixing them corrupts the baseline." So the archive below physically refuses
# in-place edits, and promoting a draft into it is an explicit, audited act.

import json, math, re, textwrap
from collections import Counter
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

from edss_core import (DOMAINS, KPI_DEFINITIONS, REPORTING_PERIOD, ToolRouter,
                       Connector, validate_kpi_result)


class ArchiveImmutableError(Exception):
    """Raised on any attempt to edit a record that is already archived."""


@dataclass
class RunState:
    """Short-term working memory. One run. Thrown away at the end."""
    run_id: str
    period: str
    draft: Dict[str, Any] = field(default_factory=dict)
    scratch: Dict[str, Any] = field(default_factory=dict)
    retrieved: List[Dict[str, Any]] = field(default_factory=list)

    def note(self, key: str, value: Any) -> None:
        self.scratch[key] = value

    def summary(self) -> str:
        return (f"RunState({self.run_id}) draft_keys={sorted(self.draft)} "
                f"scratch_keys={sorted(self.scratch)} retrieved={len(self.retrieved)}")


class Archive:
    """Long-term store. Append-only; every write is versioned and timestamped."""

    def __init__(self):
        self._records: Dict[str, Dict[str, Any]] = {}
        self.writes: List[Dict[str, Any]] = []

    # ---- writing -------------------------------------------------------
    def put(self, record_id: str, record: Dict[str, Any]) -> Dict[str, Any]:
        if record_id in self._records:
            raise ArchiveImmutableError(
                f"'{record_id}' is already archived; the archive is append-only. "
                f"Write a new version id instead of editing the baseline.")
        stored = {**record, "record_id": record_id,
                  "archived_at": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        self._records[record_id] = stored
        self.writes.append({"record_id": record_id, "tags": record.get("tags", []),
                            "at": stored["archived_at"]})
        return stored

    # ---- reading -------------------------------------------------------
    def get(self, record_id: str) -> Dict[str, Any]:
        # A COPY, so a caller mutating what it read cannot corrupt the baseline.
        return json.loads(json.dumps(self._records[record_id]))

    def all(self) -> List[Dict[str, Any]]:
        return [self.get(rid) for rid in self._records]

    def __len__(self):
        return len(self._records)

    def __contains__(self, record_id):
        return record_id in self._records


print("Memory split declared:")
print("  RunState — mutable, one run, never a source of truth")
print("  Archive  — append-only, versioned, returns copies so readers cannot corrupt it")


Memory split declared:
  RunState — mutable, one run, never a source of truth
  Archive  — append-only, versioned, returns copies so readers cannot corrupt it


### Cell 3 · What is in the archive

Two record types, both tagged:

* **`decision_brief`** — four past leadership briefs, each with the decisions taken **and what happened next**. The outcome field is what turns an archive into something worth querying.
* **`kpi_snapshot`** — closed-year institution KPI values for 2023-24 and 2024-25, the baseline for trend.

In [3]:
# =============================================================================
# Cell 3  ·  What is in the archive
# =============================================================================
# Two kinds of long-term record, both tagged so they can be retrieved by topic:
#
#   decision_brief  — a past leadership brief, its decisions and what happened
#   kpi_snapshot    — a closed year's institution KPI values, for trend context
#
# These are the "history of prior decision / KPIs" Chapter 15 lists under Lab 4.

archive = Archive()

PRIOR_BRIEFS = [
    {"record_id": "BRIEF-2023-24-Q4",
     "type": "decision_brief", "period": "2023-24",
     "title": "Placement recovery after the 2023 hiring slowdown",
     "tags": ["placement", "core-branches", "recruiter-outreach", "below-target"],
     "summary": "Placement closed at 71.80% against an 80% target. Core branches drove "
                "the gap; circuit branches held. The Council approved a recruiter "
                "outreach drive and deferred the package target.",
     "decisions": ["Approved INR 12 lakh recruiter outreach budget",
                   "Deferred the average-package target by one year"],
     "outcome": "Placement rose to 73.50% the following year — short of target but "
                "the trend reversed. The wider recruiter pool brought in lower-paying "
                "offers, and average package began to slide."},
    {"record_id": "BRIEF-2024-25-Q2",
     "type": "decision_brief", "period": "2024-25",
     "title": "Research output against the NIRF submission",
     "tags": ["research", "publications", "citations", "nirf", "below-target"],
     "summary": "Publications per faculty stood at 1.05 against a 1.50 target and "
                "citations per faculty at 6.40. The Council approved a seed-grant "
                "scheme and asked for a per-department breakdown each quarter.",
     "decisions": ["Approved a seed-grant scheme for research-active faculty",
                   "Made department-level drill-down mandatory for research KPIs"],
     "outcome": "Publications rose to 1.12; the reporting change stuck and is now "
                "standing practice."},
    {"record_id": "BRIEF-2024-25-Q4",
     "type": "decision_brief", "period": "2024-25",
     "title": "Budget utilisation and the unreconciled Q4 position",
     "tags": ["finance", "budget-utilisation", "provisional", "reconciliation"],
     "summary": "Utilisation was reported at 88.10% but Q4 was unreconciled at the time "
                "of review. The Council declined to act on a provisional figure and "
                "deferred the decision to the post-audit cycle.",
     "decisions": ["Deferred all budget reallocation until the books closed",
                   "Required provisional figures to be labelled on the face of the brief"],
     "outcome": "Final utilisation came in at 86.90% — 1.2 points below the provisional "
                "figure, vindicating the deferral."},
    {"record_id": "BRIEF-2024-25-Q3",
     "type": "decision_brief", "period": "2024-25",
     "title": "Pass percentage and the rounding dispute",
     "tags": ["academic", "pass-percentage", "rounding", "at-risk"],
     "summary": "Pass percentage was circulated as '92%' when the unrounded figure was "
                "91.60%, which is below the 92.00% target. The Council directed that "
                "KPI figures be reported unrounded.",
     "decisions": ["Directed that all KPI figures be reported unrounded",
                   "Ordered remedial coaching in the two weakest departments"],
     "outcome": "The reporting rule is now enforced by the KPI engine itself."},
]

PRIOR_KPI_SNAPSHOTS = [
    {"record_id": "KPI-SNAPSHOT-2023-24", "type": "kpi_snapshot", "period": "2023-24",
     "title": "Institution KPI snapshot 2023-24",
     "tags": ["kpi-snapshot", "closed-year", "baseline"],
     "values": {"KPI-A1": 90.40, "KPI-A2": 2.38, "KPI-R1": 0.98, "KPI-R2": 5.90,
                "KPI-R3": 198.0, "KPI-P1": 71.80, "KPI-P2": 7.10, "KPI-P3": 64.20,
                "KPI-F1": 84.30}},
    {"record_id": "KPI-SNAPSHOT-2024-25", "type": "kpi_snapshot", "period": "2024-25",
     "title": "Institution KPI snapshot 2024-25",
     "tags": ["kpi-snapshot", "closed-year", "baseline"],
     "values": {"KPI-A1": 91.20, "KPI-A2": 2.44, "KPI-R1": 1.12, "KPI-R2": 6.40,
                "KPI-R3": 221.5, "KPI-P1": 73.50, "KPI-P2": 7.95, "KPI-P3": 69.10,
                "KPI-F1": 86.90}},
]

for rec in PRIOR_BRIEFS + PRIOR_KPI_SNAPSHOTS:
    archive.put(rec["record_id"], {k: v for k, v in rec.items() if k != "record_id"})

print(f"Archive loaded: {len(archive)} records")
for r in archive.all():
    print(f"  {r['record_id']:<24} {r['type']:<15} {r['period']:<9} "
          f"tags={r['tags']}")


Archive loaded: 6 records
  BRIEF-2023-24-Q4         decision_brief  2023-24   tags=['placement', 'core-branches', 'recruiter-outreach', 'below-target']
  BRIEF-2024-25-Q2         decision_brief  2024-25   tags=['research', 'publications', 'citations', 'nirf', 'below-target']
  BRIEF-2024-25-Q4         decision_brief  2024-25   tags=['finance', 'budget-utilisation', 'provisional', 'reconciliation']
  BRIEF-2024-25-Q3         decision_brief  2024-25   tags=['academic', 'pass-percentage', 'rounding', 'at-risk']
  KPI-SNAPSHOT-2023-24     kpi_snapshot    2023-24   tags=['kpi-snapshot', 'closed-year', 'baseline']
  KPI-SNAPSHOT-2024-25     kpi_snapshot    2024-25   tags=['kpi-snapshot', 'closed-year', 'baseline']


### Cell 4 · Retrieval by tag

Exact and cheap — the right tool when you know what you are looking for. `any` mode unions, `all` mode requires every tag. A tag nobody carries returns nothing rather than a nearest guess.

In [4]:
# =============================================================================
# Cell 4  ·  Retrieval by tag — the precise path
# =============================================================================
# Tags answer "give me everything about placement". Exact, cheap, and the right
# tool when you know what you are looking for.

def retrieve_by_tags(store: Archive, tags: List[str], mode: str = "any",
                     record_type: Optional[str] = None) -> List[Dict[str, Any]]:
    """mode='any' -> union; mode='all' -> records carrying every tag."""
    if mode not in ("any", "all"):
        raise ValueError("mode must be 'any' or 'all'")
    wanted = {t.lower() for t in tags}
    hits = []
    for rec in store.all():
        if record_type and rec.get("type") != record_type:
            continue
        have = {t.lower() for t in rec.get("tags", [])}
        overlap = wanted & have
        if (mode == "any" and overlap) or (mode == "all" and wanted <= have):
            hits.append({**rec, "matched_tags": sorted(overlap),
                         "tag_score": len(overlap) / len(wanted)})
    return sorted(hits, key=lambda r: -r["tag_score"])


print("=" * 84)
print("TAG RETRIEVAL")
print("=" * 84)
for label, tags, mode in [("anything about placement", ["placement"], "any"),
                          ("below-target items in any domain", ["below-target"], "any"),
                          ("research AND below-target", ["research", "below-target"], "all"),
                          ("finance AND provisional", ["finance", "provisional"], "all")]:
    hits = retrieve_by_tags(archive, tags, mode)
    print(f"\n  query: {label}   (tags={tags}, mode={mode})")
    for h in hits:
        print(f"     {h['record_id']:<24} matched={h['matched_tags']}  "
              f"score={h['tag_score']:.2f}")
    if not hits:
        print("     (no records)")


TAG RETRIEVAL

  query: anything about placement   (tags=['placement'], mode=any)
     BRIEF-2023-24-Q4         matched=['placement']  score=1.00

  query: below-target items in any domain   (tags=['below-target'], mode=any)
     BRIEF-2023-24-Q4         matched=['below-target']  score=1.00
     BRIEF-2024-25-Q2         matched=['below-target']  score=1.00

  query: research AND below-target   (tags=['research', 'below-target'], mode=all)
     BRIEF-2024-25-Q2         matched=['below-target', 'research']  score=1.00

  query: finance AND provisional   (tags=['finance', 'provisional'], mode=all)
     BRIEF-2024-25-Q4         matched=['finance', 'provisional']  score=1.00


### Cell 5 · Retrieval by similarity

Tags fail when you don't know the right tag. Similarity answers *"have we seen anything like this before?"* — which is what the coordinator actually needs when holding a fresh draft.

A small **TF-IDF + cosine** index, written out in full. No external model, no network, no API key, and every score reproducible by hand. Note the second query: an unrelated question scores **0.0000** rather than returning the least-bad match with false confidence.

In [5]:
# =============================================================================
# Cell 5  ·  Retrieval by similarity — the fuzzy path
# =============================================================================
# Tags fail when you do not know the right tag. Similarity answers "have we
# seen anything LIKE this before?" — which is what the coordinator actually
# needs when it is holding a fresh draft.
#
# A small TF-IDF + cosine index, written out in full: no external model, no
# network, no API key, and every score reproducible by hand.

STOPWORDS = {"the", "a", "an", "and", "or", "of", "to", "in", "on", "at", "for", "was",
             "were", "is", "are", "be", "been", "by", "with", "that", "this", "it",
             "as", "from", "but", "not", "no", "we", "our", "its", "their", "has",
             "had", "have", "than", "then", "so", "which", "when", "all", "one"}


def tokenize(text: str) -> List[str]:
    return [t for t in re.findall(r"[a-z0-9\-]+", text.lower())
            if t not in STOPWORDS and len(t) > 2]


class SimilarityIndex:
    """TF-IDF vectors + cosine similarity over the archive's text."""

    def __init__(self, records: List[Dict[str, Any]]):
        self.records = records
        self.docs = [self._text(r) for r in records]
        self.tokens = [tokenize(d) for d in self.docs]
        n = len(self.tokens)
        df = Counter(t for toks in self.tokens for t in set(toks))
        # Smoothed IDF, so a term in every document still carries a little weight.
        self.idf = {t: math.log((n + 1) / (c + 1)) + 1.0 for t, c in df.items()}
        self.vectors = [self._vector(toks) for toks in self.tokens]

    @staticmethod
    def _text(rec: Dict[str, Any]) -> str:
        parts = [rec.get("title", ""), rec.get("summary", ""),
                 " ".join(rec.get("tags", [])), " ".join(rec.get("decisions", [])),
                 rec.get("outcome", "")]
        return " ".join(p for p in parts if p)

    def _vector(self, toks: List[str]) -> Dict[str, float]:
        if not toks:
            return {}
        tf = Counter(toks)
        vec = {t: (c / len(toks)) * self.idf.get(t, 1.0) for t, c in tf.items()}
        norm = math.sqrt(sum(v * v for v in vec.values())) or 1.0
        return {t: v / norm for t, v in vec.items()}

    @staticmethod
    def _cosine(a: Dict[str, float], b: Dict[str, float]) -> float:
        small, large = (a, b) if len(a) < len(b) else (b, a)
        return sum(v * large.get(t, 0.0) for t, v in small.items())

    def query(self, text: str, k: int = 3,
              record_type: Optional[str] = None) -> List[Dict[str, Any]]:
        qv = self._vector(tokenize(text))
        scored = []
        for rec, vec in zip(self.records, self.vectors):
            if record_type and rec.get("type") != record_type:
                continue
            scored.append({**rec, "similarity": round(self._cosine(qv, vec), 4)})
        return sorted(scored, key=lambda r: -r["similarity"])[:k]


index = SimilarityIndex(archive.all())
print(f"Similarity index built over {len(archive)} records, "
      f"{len(index.idf)} distinct terms.\n")

print("=" * 84)
print("SIMILARITY RETRIEVAL — 'have we seen anything like this before?'")
print("=" * 84)
DRAFT_TEXT = ("Placement percentage is 75.12% against an 80% target for 2025-26. "
              "The core branches are far below the circuit branches and are pulling "
              "the institution figure down.")
print(f"\n  current draft:\n{textwrap.indent(textwrap.fill(DRAFT_TEXT, 74), '    ')}\n")
for h in index.query(DRAFT_TEXT, k=3, record_type="decision_brief"):
    print(f"     {h['similarity']:.4f}  {h['record_id']:<24} {h['title']}")

print("\n  A query with no shared vocabulary returns low scores, not false confidence:")
for h in index.query("hostel mess menu and canteen hygiene inspection", k=3,
                     record_type="decision_brief"):
    print(f"     {h['similarity']:.4f}  {h['record_id']:<24} {h['title'][:48]}")


Similarity index built over 6 records, 129 distinct terms.

SIMILARITY RETRIEVAL — 'have we seen anything like this before?'

  current draft:
    Placement percentage is 75.12% against an 80% target for 2025-26. The core
    branches are far below the circuit branches and are pulling the
    institution figure down.

     0.3931  BRIEF-2023-24-Q4         Placement recovery after the 2023 hiring slowdown
     0.1516  BRIEF-2024-25-Q3         Pass percentage and the rounding dispute
     0.0734  BRIEF-2024-25-Q4         Budget utilisation and the unreconciled Q4 position

  A query with no shared vocabulary returns low scores, not false confidence:
     0.0000  BRIEF-2023-24-Q4         Placement recovery after the 2023 hiring slowdow
     0.0000  BRIEF-2024-25-Q2         Research output against the NIRF submission
     0.0000  BRIEF-2024-25-Q4         Budget utilisation and the unreconciled Q4 posit


### Cell 6 · Hybrid retrieval, and the no-repeat check

Tags narrow, similarity ranks. Used together you get *the most relevant placement item*, which is neither query alone.

Then the use that matters: before the brief proposes an option, it asks the archive whether the Council has **already tried it**. One proposal comes back as already-approved-in-2023-24 with its outcome; the other as genuinely new. A brief that re-proposes a failed remedy as fresh thinking is worse than no brief.

In [6]:
# =============================================================================
# Cell 6  ·  Hybrid retrieval, and the no-repeat check
# =============================================================================
# Tags narrow; similarity ranks. Used together you get "the most relevant
# placement item", which is neither query alone.

def retrieve(store: Archive, idx: SimilarityIndex, text: str,
             tags: Optional[List[str]] = None, k: int = 3,
             record_type: Optional[str] = None) -> List[Dict[str, Any]]:
    """Tag filter first (precision), then similarity rank (relevance)."""
    pool = (retrieve_by_tags(store, tags, "any", record_type) if tags
            else [r for r in store.all()
                  if not record_type or r.get("type") == record_type])
    if not pool:
        return []
    sub = SimilarityIndex(pool)
    hits = sub.query(text, k=k, record_type=record_type)
    by_id = {h["record_id"]: h for h in hits}
    for p in pool:
        if p["record_id"] in by_id and "tag_score" in p:
            by_id[p["record_id"]]["tag_score"] = p["tag_score"]
    return hits


print("=" * 84)
print("HYBRID RETRIEVAL — tag filter, then similarity rank")
print("=" * 84)
hybrid = retrieve(archive, index, DRAFT_TEXT, tags=["placement", "below-target"],
                  k=3, record_type="decision_brief")
for h in hybrid:
    print(f"  {h['similarity']:.4f}  {h['record_id']:<24} {h['title']}")

# ---- the no-repeat check: has leadership already decided this? -------------
print("\n" + "=" * 84)
print("NO-REPEAT CHECK — is a proposed option something the Council already tried?")
print("=" * 84)
PROPOSALS = [
    "Approve a recruiter outreach budget to lift placement in the core branches.",
    "Introduce a credit-bearing mandatory summer internship for all eligible students.",
]
for proposal in PROPOSALS:
    prior = index.query(proposal, k=1, record_type="decision_brief")[0]
    seen = prior["similarity"] >= 0.20
    print(f"\n  proposal: {textwrap.fill(proposal, 70, subsequent_indent='            ')}")
    print(f"     closest prior decision : {prior['record_id']} "
          f"(similarity {prior['similarity']:.4f})")
    if seen:
        print(f"     -> ALREADY TRIED in {prior['period']}: {prior['decisions'][0]}")
        print(f"        outcome: {textwrap.fill(prior['outcome'], 64, subsequent_indent='                 ')}")
        print(f"        The brief must say so rather than re-proposing it as new.")
    else:
        print(f"     -> no close prior decision; this is a genuinely new proposal.")


HYBRID RETRIEVAL — tag filter, then similarity rank
  0.3865  BRIEF-2023-24-Q4         Placement recovery after the 2023 hiring slowdown
  0.0512  BRIEF-2024-25-Q2         Research output against the NIRF submission

NO-REPEAT CHECK — is a proposed option something the Council already tried?

  proposal: Approve a recruiter outreach budget to lift placement in the core
            branches.
     closest prior decision : BRIEF-2023-24-Q4 (similarity 0.5556)
     -> ALREADY TRIED in 2023-24: Approved INR 12 lakh recruiter outreach budget
        outcome: Placement rose to 73.50% the following year — short of target
                 but the trend reversed. The wider recruiter
                 pool brought in lower-paying offers, and
                 average package began to slide.
        The brief must say so rather than re-proposing it as new.

  proposal: Introduce a credit-bearing mandatory summer internship for all
            eligible students.
     closest prior decision : BRIEF-20

### Cell 7 · What the archive is actually for — year-on-year trend

Lab 1's revision round asked for "year-on-year trend context" and the plan could only *promise* it. The archive makes it deliverable: the current year is computed from live tools, the prior years are read from closed snapshots, and the two are never mixed.

The payoff is the last block. **Average package is below target *and falling*** — and read against the archived 2023-24 decision, the cause is visible: the approved recruiter-outreach drive widened the pool and lifted placement percentage, and the new recruiters pay less.

Two KPIs moving in opposite directions because of one past decision. That finding exists only because the decision **and** the baseline were both archived.

In [7]:
# =============================================================================
# Cell 7  ·  What the archive is actually FOR — year-on-year trend
# =============================================================================
# Lab 1's revision round asked for "year-on-year trend context" and the plan
# could only promise it. The archive is what makes it deliverable: the current
# year is computed from live tools, the prior years are read from the closed
# snapshots, and the two are never mixed.

router = ToolRouter(agent="Lab 4 memory runner")
connector = Connector(router)
rows = connector.read("department reports",
                      reason="current-year KPIs for the trend comparison",
                      period=REPORTING_PERIOD)["rows"]

current = {}
for kpi_id in KPI_DEFINITIONS:
    computed = router.call("compute_kpi", reason=f"{kpi_id}: current year",
                           kpi_id=kpi_id, rows=rows)
    compared = router.call("compare_to_target", reason=f"{kpi_id}: vs target",
                           kpi_id=kpi_id, value=computed["value"])
    current[kpi_id] = {"value": computed["value"], "target": compared["target"],
                       "status": compared["status"], "unit": computed["unit"],
                       "name": computed["name"]}

snap_23 = archive.get("KPI-SNAPSHOT-2023-24")["values"]
snap_24 = archive.get("KPI-SNAPSHOT-2024-25")["values"]

print("=" * 92)
print("THREE-YEAR TREND — current year from live tools, prior years from the archive")
print("=" * 92)
print(f"  {'KPI':<9}{'Name':<27}{'2023-24':>10}{'2024-25':>10}{'2025-26':>10}"
      f"{'YoY':>9}{'Target':>9}  direction")
trend = {}
for kpi_id, cur in current.items():
    a, b, c = snap_23[kpi_id], snap_24[kpi_id], cur["value"]
    yoy = c - b
    arrow = "improving" if yoy > 0 else ("flat" if abs(yoy) < 1e-9 else "declining")
    trend[kpi_id] = {"2023-24": a, "2024-25": b, "2025-26": round(c, 2),
                     "yoy": round(yoy, 2), "direction": arrow,
                     "target": cur["target"], "status": cur["status"]}
    print(f"  {kpi_id:<9}{cur['name']:<27}{a:>10.2f}{b:>10.2f}{c:>10.2f}"
          f"{yoy:>+9.2f}{cur['target']:>9.2f}  {arrow}")

improving_but_short = [k for k, t in trend.items()
                       if t["direction"] == "improving" and t["status"] != "on target"]
declining = [k for k, t in trend.items() if t["direction"] == "declining"]
print(f"\n  Improving but still below target : {improving_but_short}")
print(f"  Declining year on year           : {declining or 'none'}")
print("\n  This distinction is the whole point of the archive: 'below target' and")
print("  'getting worse' are different findings and call for different decisions.")
print("  Without a baseline the brief can only report the first.")

if declining:
    d = declining[0]
    print(f"\n  Worked example — {d} ({trend[d]['name'] if 'name' in trend[d] else current[d]['name']}):")
    print(f"     {trend[d]['2023-24']} -> {trend[d]['2024-25']} -> {trend[d]['2025-26']} "
          f"({trend[d]['yoy']:+.2f} year on year)")
    print("     Read against the 2023-24 brief in the archive, the cause is visible: the")
    print("     approved recruiter-outreach drive widened the pool and lifted placement")
    print("     percentage, and the new recruiters pay less. Two KPIs moving in opposite")
    print("     directions because of one past decision — a finding that exists only")
    print("     because the decision and the baseline were both archived.")


THREE-YEAR TREND — current year from live tools, prior years from the archive
  KPI      Name                          2023-24   2024-25   2025-26      YoY   Target  direction
  KPI-A1   Institution pass percentage     90.40     91.20     91.85    +0.65    92.00  improving
  KPI-A2   CO/PO attainment index           2.38      2.44      2.48    +0.04     2.50  improving
  KPI-R1   Publications per faculty         0.98      1.12      1.19    +0.07     1.50  improving
  KPI-R2   Citations per faculty            5.90      6.40      7.26    +0.86     9.00  improving
  KPI-R3   Grants sanctioned              198.00    221.50    242.50   +21.00   300.00  improving
  KPI-P1   Placement percentage            71.80     73.50     75.12    +1.62    80.00  improving
  KPI-P2   Average package                  7.10      7.95      7.67    -0.28     8.00  declining
  KPI-P3   Internship completion           64.20     69.10     73.38    +4.28    85.00  improving
  KPI-F1   Budget utilisation           

### Cell 8 · The pitfall, guarded

Four ways run state could corrupt the baseline, all blocked: editing the draft, mutating a retrieved record, overwriting an archived record, and promoting an unapproved draft.

Only the last path succeeds, and only after approval — because the archive is the institutional baseline, not a scratchpad.

In [8]:
# =============================================================================
# Cell 8  ·  The pitfall, guarded — run state must not touch the archive
# =============================================================================
# "Don't conflate run state with the archive — mixing them corrupts the
# baseline." Four ways that could happen, all blocked.

state = RunState(run_id="RUN-2025-26-01", period=REPORTING_PERIOD)
state.draft = {"title": "Executive decision brief — 2025-26",
               "status": "DRAFT", "kpis": {k: v["value"] for k, v in current.items()}}
state.retrieved = hybrid
state.note("trend", trend)
state.note("prior_decision_hits", [h["record_id"] for h in hybrid])

print("=" * 84)
print("SEPARATION CHECKS")
print("=" * 84)
print(f"  {state.summary()}\n")

baseline_before = archive.get("KPI-SNAPSHOT-2024-25")["values"]["KPI-P1"]

# 1 — editing the draft cannot touch the baseline
state.draft["kpis"]["KPI-P1"] = 99.99
after = archive.get("KPI-SNAPSHOT-2024-25")["values"]["KPI-P1"]
print(f"  1. draft edited to a nonsense value; archived 2024-25 KPI-P1 "
      f"{baseline_before} -> {after}   {'UNCHANGED' if after == baseline_before else 'CORRUPTED'}")
state.draft["kpis"]["KPI-P1"] = current["KPI-P1"]["value"]

# 2 — what you read out of the archive is a copy
borrowed = archive.get("KPI-SNAPSHOT-2023-24")
borrowed["values"]["KPI-A1"] = 0.0
still = archive.get("KPI-SNAPSHOT-2023-24")["values"]["KPI-A1"]
print(f"  2. a retrieved record was mutated by the caller; archive still reads "
      f"{still}   {'UNCHANGED' if still == 90.40 else 'CORRUPTED'}")

# 3 — the archive is append-only
try:
    archive.put("KPI-SNAPSHOT-2024-25", {"values": {"KPI-P1": 0.0}})
    print("  3. [!!] overwrite of an archived record was ALLOWED")
except ArchiveImmutableError as exc:
    print(f"  3. overwrite refused: {exc}")

# 4 — promoting a draft into the archive is explicit, audited and versioned
def archive_run(store: Archive, st: RunState, approved_by: str) -> Dict[str, Any]:
    """The ONLY path from run state into long-term memory."""
    if st.draft.get("status") != "APPROVED":
        raise ArchiveImmutableError(
            "refusing to archive a brief that has not been approved — the archive is "
            "the institutional baseline, not a scratchpad")
    return store.put(f"BRIEF-{st.period.replace('academic year ', '')}-{st.run_id}",
                     {"type": "decision_brief", "period": st.period,
                      "title": st.draft["title"],
                      "tags": ["placement", "research", "academic", "finance",
                               "below-target"],
                      "summary": st.draft.get("summary", ""),
                      "decisions": st.draft.get("decisions", []),
                      "approved_by": approved_by})

try:
    archive_run(archive, state, "Dr. A. Principal")
    print("  4. [!!] an unapproved draft reached the archive")
except ArchiveImmutableError as exc:
    print(f"  4. unapproved draft refused: {exc}")

state.draft["status"] = "APPROVED"
state.draft["summary"] = ("9 of 9 KPIs off target; placement improving but average "
                          "package declining for the first time.")
state.draft["decisions"] = ["Noted the average-package decline against the 2023-24 "
                            "recruiter-outreach decision."]
promoted = archive_run(archive, state, "Dr. A. Principal")
print(f"  5. approved brief archived as {promoted['record_id']} at "
      f"{promoted['archived_at']}")
print(f"     archive is now {len(archive)} records; "
      f"{len(archive.writes)} audited writes")


SEPARATION CHECKS
  RunState(RUN-2025-26-01) draft_keys=['kpis', 'status', 'title'] scratch_keys=['prior_decision_hits', 'trend'] retrieved=2

  1. draft edited to a nonsense value; archived 2024-25 KPI-P1 73.5 -> 73.5   UNCHANGED
  2. a retrieved record was mutated by the caller; archive still reads 90.4   UNCHANGED
  3. overwrite refused: 'KPI-SNAPSHOT-2024-25' is already archived; the archive is append-only. Write a new version id instead of editing the baseline.
  4. unapproved draft refused: refusing to archive a brief that has not been approved — the archive is the institutional baseline, not a scratchpad
  5. approved brief archived as BRIEF-2025-26-RUN-2025-26-01 at 2026-09-28T20:13:07+00:00
     archive is now 7 records; 7 audited writes


### Cell 9 · Acceptance checks — the rubric, executed

Thirteen checks covering both halves of the acceptance line (correctly-tagged candidates **and** similar prior items), the no-repeat use, and all four separation guarantees. **D13** is the one worth reading: it asserts the archive produces a finding the current year alone cannot.

In [9]:
# =============================================================================
# Cell 9  ·  Acceptance checks — the rubric, executed
# =============================================================================
# Worklet 15, Lab 4 acceptance: "For a given task, retrieval returns
# correctly-tagged candidates and similar prior items."
# Pitfall guarded: run state and the archive must not be conflated.

checks = []


def check(name, condition, evidence=""):
    checks.append((name, bool(condition), evidence))


# --- retrieval: tags ---------------------------------------------------
placement_hits = retrieve_by_tags(archive, ["placement"], "any", "decision_brief")
check("D1  tag retrieval returns correctly-tagged candidates",
      placement_hits and all("placement" in [t.lower() for t in h["tags"]]
                             for h in placement_hits),
      f"{len(placement_hits)} record(s), every one carrying the 'placement' tag")

both = retrieve_by_tags(archive, ["research", "below-target"], "all")
check("D2  'all' mode requires every tag, not any",
      all({"research", "below-target"} <= {t.lower() for t in h["tags"]} for h in both)
      and len(both) < len(retrieve_by_tags(archive, ["research", "below-target"], "any")),
      f"'all' -> {len(both)} record(s); 'any' -> "
      f"{len(retrieve_by_tags(archive, ['research', 'below-target'], 'any'))}")

check("D3  a tag nobody carries returns nothing, not a wrong guess",
      retrieve_by_tags(archive, ["hostel-mess"], "any") == [],
      "retrieve_by_tags(['hostel-mess']) -> []")

# --- retrieval: similarity ---------------------------------------------
top = index.query(DRAFT_TEXT, k=1, record_type="decision_brief")[0]
check("D4  similarity retrieval surfaces the right prior item",
      top["record_id"] == "BRIEF-2023-24-Q4" and top["similarity"] > 0.2,
      f"top hit {top['record_id']} at similarity {top['similarity']:.4f}")

ranked = index.query(DRAFT_TEXT, k=4, record_type="decision_brief")
check("D5  similarity is ordered, not arbitrary",
      all(ranked[i]["similarity"] >= ranked[i + 1]["similarity"]
          for i in range(len(ranked) - 1)),
      f"scores: {[round(r['similarity'], 3) for r in ranked]}")

unrelated = index.query("hostel mess menu and canteen hygiene", k=1,
                        record_type="decision_brief")[0]
check("D6  an unrelated query scores ~0 rather than returning false confidence",
      unrelated["similarity"] < 0.05,
      f"unrelated query top similarity {unrelated['similarity']:.4f}")

check("D7  hybrid retrieval respects the tag filter AND ranks by similarity",
      hybrid and all("placement" in [t.lower() for t in h["tags"]]
                     or "below-target" in [t.lower() for t in h["tags"]]
                     for h in hybrid)
      and hybrid[0]["similarity"] >= hybrid[-1]["similarity"],
      f"{len(hybrid)} hits, top = {hybrid[0]['record_id']}")

# --- the no-repeat use --------------------------------------------------
repeat = index.query(PROPOSALS[0], k=1, record_type="decision_brief")[0]
novel = index.query(PROPOSALS[1], k=1, record_type="decision_brief")[0]
check("D8  a previously-taken decision is recognised; a new one is not flagged",
      repeat["similarity"] >= 0.20 > novel["similarity"],
      f"already-tried proposal {repeat['similarity']:.3f} vs new proposal "
      f"{novel['similarity']:.3f}")

# --- the split -----------------------------------------------------------
check("D9  editing run state cannot change the archived baseline",
      archive.get("KPI-SNAPSHOT-2024-25")["values"]["KPI-P1"] == 73.50,
      "archived 2024-25 KPI-P1 still 73.50 after the draft was overwritten")

check("D10 the archive hands out copies, so a reader cannot corrupt it",
      archive.get("KPI-SNAPSHOT-2023-24")["values"]["KPI-A1"] == 90.40,
      "a mutated retrieved record left the store untouched")

check("D11 the archive is append-only",
      "KPI-SNAPSHOT-2024-25" in archive and len(archive.writes) == len(archive),
      f"{len(archive.writes)} writes for {len(archive)} records — no overwrites")

check("D12 only an approved draft can enter long-term memory",
      any(r.get("approved_by") for r in archive.all()),
      "the unapproved draft was refused; the approved one carries approved_by")

# --- the payoff ----------------------------------------------------------
check("D13 the archive enables a finding the current year alone cannot produce",
      declining and trend[declining[0]]["yoy"] < 0,
      f"{declining[0]} is below target AND declining "
      f"({trend[declining[0]]['yoy']:+.2f} YoY) — invisible without a baseline")

print("=" * 84)
print("LAB 4 ACCEPTANCE CHECKS")
print("=" * 84)
for name, ok, ev in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}]  {name}")
    if ev:
        print(f"          {ev}")
failed = [n for n, ok, _ in checks if not ok]
print("=" * 84)
print(f"  {len(checks) - len(failed)}/{len(checks)} checks passed.")
assert not failed, f"FAILED: {failed}"
print("  Lab 4 acceptance criteria satisfied.")


LAB 4 ACCEPTANCE CHECKS
  [PASS]  D1  tag retrieval returns correctly-tagged candidates
          2 record(s), every one carrying the 'placement' tag
  [PASS]  D2  'all' mode requires every tag, not any
          'all' -> 2 record(s); 'any' -> 3
  [PASS]  D3  a tag nobody carries returns nothing, not a wrong guess
          retrieve_by_tags(['hostel-mess']) -> []
  [PASS]  D4  similarity retrieval surfaces the right prior item
          top hit BRIEF-2023-24-Q4 at similarity 0.3931
  [PASS]  D5  similarity is ordered, not arbitrary
          scores: [0.393, 0.152, 0.073, 0.061]
  [PASS]  D6  an unrelated query scores ~0 rather than returning false confidence
          unrelated query top similarity 0.0000
  [PASS]  D7  hybrid retrieval respects the tag filter AND ranks by similarity
          2 hits, top = BRIEF-2023-24-Q4
  [PASS]  D8  a previously-taken decision is recognised; a new one is not flagged
          already-tried proposal 0.556 vs new proposal 0.079
  [PASS]  D9  editing 

### Cell 10 · Deliverables

In [10]:
# =============================================================================
# Cell 10 · Deliverables
# =============================================================================
from pathlib import Path

outdir = Path("lab4_artifacts")
outdir.mkdir(exist_ok=True)

artifacts = {
    "lab4_archive.json": archive.all(),
    "lab4_archive_write_log.json": archive.writes,
    "lab4_run_state.json": {"run_id": state.run_id, "period": state.period,
                            "draft": state.draft, "scratch_keys": sorted(state.scratch),
                            "retrieved": [h["record_id"] for h in state.retrieved]},
    "lab4_trend.json": trend,
    "lab4_retrieval_examples.json": {
        "tag_query_placement": [h["record_id"] for h in placement_hits],
        "similarity_query": {"text": DRAFT_TEXT,
                             "hits": [{"record_id": h["record_id"],
                                       "similarity": h["similarity"]}
                                      for h in index.query(DRAFT_TEXT, k=4,
                                                           record_type="decision_brief")]},
        "hybrid_query": [{"record_id": h["record_id"], "similarity": h["similarity"]}
                         for h in hybrid],
        "no_repeat_check": [{"proposal": p,
                             "closest": index.query(p, k=1,
                                                    record_type="decision_brief")[0]["record_id"],
                             "similarity": index.query(p, k=1,
                                                       record_type="decision_brief")[0]["similarity"]}
                            for p in PROPOSALS]},
    "lab4_tool_trace.json": router.trace,
    "lab4_acceptance_checks.json": [{"check": n, "passed": ok, "evidence": e}
                                    for n, ok, e in checks],
}
for fname, obj in artifacts.items():
    (outdir / fname).write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str))

print("=" * 84)
print("ARTIFACTS WRITTEN")
print("=" * 84)
for fname in artifacts:
    print(f"  {outdir / fname}  ({(outdir / fname).stat().st_size:,} bytes)")

print(f"\n  Archive records     : {len(archive)}  ({len(archive.writes)} audited writes)")
print(f"  Index vocabulary    : {len(index.idf)} terms over {len(archive.all())} records")
print(f"  Run-state keys      : draft={sorted(state.draft)}, scratch={sorted(state.scratch)}")
print(f"  Acceptance checks   : {sum(ok for _, ok, _ in checks)}/{len(checks)} passed")


ARTIFACTS WRITTEN
  lab4_artifacts/lab4_archive.json  (4,817 bytes)
  lab4_artifacts/lab4_archive_write_log.json  (1,321 bytes)
  lab4_artifacts/lab4_run_state.json  (765 bytes)
  lab4_artifacts/lab4_trend.json  (1,596 bytes)
  lab4_artifacts/lab4_retrieval_examples.json  (1,231 bytes)
  lab4_artifacts/lab4_tool_trace.json  (15,376 bytes)
  lab4_artifacts/lab4_acceptance_checks.json  (2,208 bytes)

  Archive records     : 7  (7 audited writes)
  Index vocabulary    : 129 terms over 7 records
  Run-state keys      : draft=['decisions', 'kpis', 'status', 'summary', 'title'], scratch=['prior_decision_hits', 'trend']
  Acceptance checks   : 13/13 passed


---

## Where Lab 4 leaves the worklet

**Nodes activated:** the Retrieval node — and the seed of the validation / no-repeat screening that Labs 9–12 will build on.

**What exists now:** a coordinator that can ask *"is this getting worse?"* and *"have we tried this before?"* — and an archive that cannot be corrupted by the run using it.

**What Lab 5 adds:** one governed door. Right now the tools reach their sources directly and the archive is a Python object in the notebook. Lab 5 puts **every** source — department reports, the KPI register, external benchmarks and now the archive — behind a single typed MCP-style connector with auth and scopes, so no agent touches a source directly and swapping a source needs no agent change.

> Lab 5 pseudocode (Chapter 15):
> ```
> connector.expose:
>     read('institutional KPIs & dashboards')
>     read('department reports')
>     read('external benchmarks')
> agents.use(connector)   # no agent touches a source directly
> ```